# probe_viz_flamingo.ipynb

Audio-Flamingo-3 representation visualizations and prompt-result summaries.

Set the Flamingo representation and result paths, then run the selected visualization or summary section.

Use the setup steps in the repository README before running this notebook.


In [ ]:
import pandas as pd
#BASE_DIR = "/content/drive/MyDrive/subprobe/task3_full_run_qwen"
BASE_DIR = "/content/drive/MyDrive/subprobe/task3_full_run_flamingo"
#PRED_CSV = BASE_DIR + "/manipulated_probe_predictions.csv"
PRED_CSV = BASE_DIR + "/subtle_probe_predictions.csv"
LABEL_CSV = BASE_DIR + "/task3_results.csv"

TARGET_LAYER = "lm_2"

pred_df = pd.read_csv(PRED_CSV)
label_df = pd.read_csv(LABEL_CSV)

# clean paths (VERY IMPORTANT)
pred_df["rep_path"] = pred_df["rep_path"].astype(str).str.strip()
label_df["rep_path"] = label_df["rep_path"].astype(str).str.strip()

print("Pred:", len(pred_df))
print("Label:", len(label_df))
df = pd.merge(
    pred_df,
    label_df[["rep_path", "labels","language"]],
    on="rep_path",
    how="inner"
)
# filter layer
df = df[df["layer"] == TARGET_LAYER].copy()

# drop bad rows
df = df.dropna(subset=["rep_path"])



df["rep_path"] = df["rep_path"].astype(str).str.strip()


print("Merged:", len(df))

import os
import shutil
from tqdm import tqdm

tqdm.pandas()  # enable progress_apply

LOCAL_REP_DIR = "./local_reps"
os.makedirs(LOCAL_REP_DIR, exist_ok=True)

# def copy_local(path):
#     filename = os.path.basename(path)
#     new_path = os.path.join(LOCAL_REP_DIR, filename)

#     if not os.path.exists(new_path):
#         shutil.copy(path, new_path)

#     return new_path

# df["rep_path_local"] = df["rep_path"].progress_apply(copy_local)

import ast

def parse_labels(s):
    try:
        return ast.literal_eval(s)
    except:
        return []

df["subtle_labels"] = df["labels"].apply(parse_labels)
COARSE_MAP = {
    "anger": "ANG", "annoyance": "ANG", "disapproval": "ANG",
    "disgust": "DIS",
    "fear": "FEA", "nervousness": "FEA",
    "joy": "HAP", "amusement": "HAP", "excitement": "HAP",
    "pride": "HAP", "optimism": "HAP", "love": "HAP",
    "admiration": "HAP", "relief": "HAP", "gratitude": "HAP",
    "caring": "HAP", "desire": "HAP",
    "confusion": "NEU", "curiosity": "NEU", "realization": "NEU",
    "surprise": "SUR",
    "sadness": "SAD", "disappointment": "SAD",
    "remorse": "SAD", "grief": "SAD",
    "neutral": "NEU"
}

def get_coarse(labels):
    mapped = [COARSE_MAP.get(l, None) for l in labels]
    mapped = [m for m in mapped if m is not None]
    if not mapped:
        return "UNK"
    return max(set(mapped), key=mapped.count)
def is_cross(labels):
    mapped = [COARSE_MAP.get(l, None) for l in labels]
    mapped = [m for m in mapped if m is not None]

    # remove neutral
    mapped = [m for m in mapped if m != "NEU"]

    return len(set(mapped)) > 1

df["coarse_label"] = df["subtle_labels"].apply(get_coarse)
df["cross"] = df["subtle_labels"].apply(is_cross)

import torch
import numpy as np

def process_rep(x):
    if not torch.is_tensor(x):
        x = torch.tensor(x)

    x = x.squeeze()

    if x.dim() == 1:
        return x
    elif x.dim() == 2:
        return x.squeeze()
    elif x.dim() == 3:
        return x.mean(dim=1).squeeze()
    else:
        raise ValueError(f"Unexpected shape: {x.shape}")

def load_embedding(rep_path, layer):
    try:
        rep = torch.load(rep_path, map_location="cpu")
        if layer not in rep:
            return None
        x = rep[layer].float()
        x = process_rep(x)
        return x.numpy()
    except:
        return None



In [ ]:
df

In [ ]:
import torch

def inspect_pt(path, verbose=False):

    data = torch.load(path,weights_only=False, map_location="cpu")

    print(f"\n📦 File: {path}")

    # If wrapped dict (recommended format)
    if isinstance(data, dict) and "reps" in data:
        reps = data["reps"]
        print("Contains metadata:")
        for k in data:
            if k != "reps":
                print(f"  {k}: {data[k]}")
    else:
        reps = data

    print("\n🔑 Keys:")
    for k in reps.keys():
        print(f"  - {k}")

    print("\n📊 Tensor Info:")
    for k, v in reps.items():

        if not isinstance(v, torch.Tensor):
            print(f"{k}: ❌ Not a tensor ({type(v)})")
            continue

        print(f"{k}: shape={tuple(v.shape)}, dtype={v.dtype}")

        if verbose:
            print(f"   mean={v.mean().item():.4f}, std={v.std().item():.4f}")

            # print small tensors fully
            if v.numel() < 20:
                print(f"   values={v}")

    print("\n✅ Done.\n")

In [ ]:
def load_embedding(rep_path, layer):
    try:
        rep = torch.load(rep_path, map_location="cpu")
        if layer not in rep:
            return None
        x = rep[layer].float()
        x = process_rep(x)
        return x.numpy()
    except:
        return None

In [ ]:
from tqdm import tqdm
import numpy as np

X_en, X_zh = [], []
labels_en, labels_zh = [], []
cross_en, cross_zh = [], []

for _, row in tqdm(df.iterrows(), total=len(df)):
    emb = load_embedding(row["rep_path"], TARGET_LAYER)
    if emb is None:
        continue

    if row["language"] == "en":
        X_en.append(emb)
        labels_en.append(row["coarse_label"])
        cross_en.append(row["cross"])
    else:
        X_zh.append(emb)
        labels_zh.append(row["coarse_label"])
        cross_zh.append(row["cross"])

# stack
X_en = np.stack(X_en)
X_zh = np.stack(X_zh)

labels_en = np.array(labels_en)
labels_zh = np.array(labels_zh)

cross_en = np.array(cross_en)
cross_zh = np.array(cross_zh)

print("EN:", X_en.shape)
print("ZH:", X_zh.shape)

In [ ]:
X = X / np.linalg.norm(X, axis=1, keepdims=True)

In [ ]:
import umap
reducer = umap.UMAP(
    n_neighbors=50,
    min_dist=0.3,
    metric="cosine",
    random_state=42
)

X_2d = reducer.fit_transform(X)

In [ ]:
def build_subset(df_sub):
    X, labels = [], []

    for _, row in df_sub.iterrows():
        emb = load_embedding(row["rep_path"], TARGET_LAYER)
        if emb is None:
            continue

        X.append(emb)
        labels.append(row["coarse_label"])

    if len(X) == 0:
        return None, None

    return np.stack(X), np.array(labels)
import umap

COLOR_MAP = {
    "ANG": "#d62728",
    "HAP": "#2ca02c",
    "SAD": "#1f77b4",
    "FEA": "#9467bd",
    "SUR": "#ff7f0e",
    "DIS": "#8c564b",
    "NEU": "#7f7f7f"
}

def run_umap(X):
    reducer = umap.UMAP(
        n_neighbors=15,
        min_dist=0.1,
        metric="cosine",
        random_state=42
    )
    return reducer.fit_transform(X)

In [ ]:
import torch
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

COARSE_MAP = {
    "anger": "ANG", "annoyance": "ANG", "disapproval": "ANG",
    "disgust": "DIS",
    "fear": "FEA", "nervousness": "FEA",
    "joy": "HAP", "amusement": "HAP", "excitement": "HAP",
    "pride": "HAP", "optimism": "HAP", "love": "HAP",
    "admiration": "HAP", "relief": "HAP", "gratitude": "HAP",
    "caring": "HAP", "desire": "HAP","approval":"HAP",
    "confusion": "SUR", "curiosity": "SUR",
    "realization": "SUR", "surprise": "SUR",
    "sadness": "SAD", "disappointment": "SAD",
    "remorse": "SAD", "grief": "SAD",
    "neutral": "NEU"
}
BASIC_SET = {"anger","disgust","fear","joy","sadness","surprise","neutral"}
COARSE_PRIORITY = ["ANG", "DIS", "FEA", "SAD", "SUR", "HAP", "NEU"]

def map_coarse(labels):
    mapped = [COARSE_MAP.get(l, "NEU") for l in labels]
    for p in COARSE_PRIORITY:
        if p in mapped:
            return p
    return "NEU"


def classify_split(labels):

    if not isinstance(labels, list):
        return "invalid"

    labels = list(set(labels))

    # case 1: strict basic
    if len(labels) == 1 and labels[0] in BASIC_SET:
        return "basic"

    # case 2: fine singleton
    if len(labels) == 1:
        return "fine_single"

    # map coarse
    coarse = set(COARSE_MAP.get(l, "UNK") for l in labels)

    # 🔥 remove neutral + unknown
    coarse.discard("NEU")
    coarse.discard("UNK")

    # if nothing left (e.g., only neutral/unknown)
    if len(coarse) == 0:
        return "fine_single"  # or "neutral_only" if you want

    # cross vs intra
    if len(coarse) > 1:
        return "subtle_cross"

    return "subtle_intra"

df["coarse"] = df["subtle_labels"].apply(map_coarse)
df["split"] = df["subtle_labels"].apply(classify_split)
df["n_labels"] = df["subtle_labels"].apply(len)
def load_rep(path):
    x = torch.load(path, map_location="cpu")

    if isinstance(x, dict):
        x = x[TARGET_LAYER]

    x = torch.tensor(x)

    if x.dim() == 3:
        x = x.mean(dim=1).squeeze(0)
    elif x.dim() == 2:
        x = x.squeeze(0)

    return x.numpy()

In [ ]:
df["split"]

In [ ]:
for lang in df["language"].unique():
    df_lang = df[df["language"] == lang]

    for split in ["basic", "subtle_pure_fine", "subtle_intra", "subtle_cross"]:
        df_split = df_lang[df_lang["split"] == split]

In [ ]:
for i in range(10):
    labels = df["subtle_labels"].iloc[i]
    split = classify_split(labels)
    print(labels, "→", split)

In [ ]:
import numpy as np

def compute_centroids(X, labels):
    centroids = {}
    for l in np.unique(labels):
        centroids[l] = X[labels == l].mean(axis=0)
    return centroids
from scipy.spatial.distance import cdist

def centroid_separation_score(X, labels):
    centroids = compute_centroids(X, labels)
    classes = list(centroids.keys())

    correct = 0
    total = len(X)

    centroid_matrix = np.stack([centroids[c] for c in classes])

    # distances: [N samples × num_classes]
    dists = cdist(X, centroid_matrix, metric="cosine")

    for i in range(len(X)):
        true_label = labels[i]
        true_idx = classes.index(true_label)

        if np.argmin(dists[i]) == true_idx:
            correct += 1

    return correct / total

import numpy as np
from scipy.spatial.distance import cdist



def loo_centroid_separation_score(X, labels):

    correct = 0
    total = len(X)

    for i in range(total):

        X_train = np.delete(X, i, axis=0)
        y_train = np.delete(labels, i)

        classes = np.unique(y_train)

        centroids = compute_centroids(
            X_train,
            y_train
        )

        centroid_matrix = np.stack(
            [centroids[c] for c in classes]
        )

        dists = cdist(
            X[i:i+1],
            centroid_matrix,
            metric="cosine"
        )[0]

        pred_idx = np.argmin(dists)
        pred_label = classes[pred_idx]

        if pred_label == labels[i]:
            correct += 1

    return correct / total

In [ ]:
results = []

for lang in ["en", "zh"]:
    for split in ["basic", "subtle_intra", "subtle_cross"]:

        df_sub = df[(df["language"] == lang) & (df["split"] == split)]
        df_sub = df_sub.sample(min(len(df_sub), 800), random_state=42)

        X, labels = build_subset(df_sub)

        if X is None:
            continue

        score = loo_centroid_separation_score(X, labels)

        results.append({
            "language": lang,
            "split": split,
            "score": score
        })

for r in results:
    print(r)

In [ ]:
import numpy as np

def bootstrap_centroid_score(
    X,
    labels,
    n_boot=1000,
    random_state=42
):
    rng = np.random.default_rng(random_state)

    scores = []
    n = len(X)

    for _ in range(n_boot):
        idx = rng.choice(n, n, replace=True)

        X_boot = X[idx]
        y_boot = labels[idx]

        score = loo_centroid_separation_score(X_boot, y_boot)
        scores.append(score)

    scores = np.array(scores)

    mean = scores.mean()
    ci_low = np.percentile(scores, 2.5)
    ci_high = np.percentile(scores, 97.5)

    return {
        "mean": mean,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "all_scores": scores
    }
#result = bootstrap_centroid_score(X, labels)


In [ ]:
def permutation_test_centroid(
    X,
    labels,
    n_perm=5000,
    random_state=42
):
    rng = np.random.default_rng(random_state)

    observed = centroid_separation_score(X, labels)

    null_scores = []

    for _ in range(n_perm):
        shuffled = rng.permutation(labels)

        score = loo_centroid_separation_score(X, shuffled)
        null_scores.append(score)

    null_scores = np.array(null_scores)

    p_value = (
        np.sum(null_scores >= observed) + 1
    ) / (n_perm + 1)

    return {
        "observed": observed,
        "null_mean": null_scores.mean(),
        "null_std": null_scores.std(),
        "p_value": p_value,
        "null_scores": null_scores
    }
#result = permutation_test_centroid(X, labels)
#print(result)

In [ ]:
results = []

for split in ["basic", "subtle_intra", "subtle_cross"]:

    # combine all languages
    df_sub = df[df["split"] == split]

    df_sub = df_sub.sample(
        min(len(df_sub), 800),
        random_state=42
    )

    X, labels = build_subset(df_sub)

    if X is None:
        continue

    # leakage-safe observed score
    observed = loo_centroid_separation_score(
        X,
        labels
    )

    # bootstrap CI
    boot = bootstrap_centroid_score(
        X,
        labels,
        n_boot=100
    )

    # permutation significance
    perm = permutation_test_centroid(
        X,
        labels,
        n_perm=100
    )

    results.append({

        "split": split,

        "score": observed,

        "ci_low": boot["ci_low"],
        "ci_high": boot["ci_high"],

        "null_mean": perm["null_mean"],
        "null_std": perm["null_std"],

        "p_value": perm["p_value"]
    })

# print nicely
for r in results:

    print(
        f"{r['split']} | "
        f"score={r['score']:.3f} | "
        f"95%CI=[{r['ci_low']:.3f}, "
        f"{r['ci_high']:.3f}] | "
        f"null={r['null_mean']:.3f}±"
        f"{r['null_std']:.3f} | "
        f"p={r['p_value']:.5f}"
    )

# save
import pandas as pd

results_df = pd.DataFrame(results)

results_df.to_csv(
    "flamingo.csv",
    index=False
)

print("Saved flamingo.csv")

In [ ]:
perm

In [ ]:
# print nicely
for r in results:

    print(
        f"{r['language']} | "
        f"{r['split']} | "
        f"score={r['score']:.3f} | "
        f"95%CI=[{r['ci_low']:.3f}, "
        f"{r['ci_high']:.3f}] | "
        f"null={r['null_mean']:.3f}±"
        f"{r['null_std']:.3f} | "
        f"p={r['p_value']:.5f}"
    )

In [ ]:
import pandas as pd

# convert results to dataframe
results_df = pd.DataFrame(results)

# save csv
results_df.to_csv(
    "flamingo.csv",
    index=False
)

print("Saved to flamingo.csv")

In [ ]:
from collections import Counter

single_counts = Counter()
multi_counts = Counter()

for labels in df["subtle_labels"]:
    if len(labels) == 1:
        single_counts[labels[0]] += 1
    else:
        for l in labels:
            multi_counts[l] += 1

valid_base_labels = [
    l for l in single_counts
    if single_counts[l] > 20 and multi_counts[l] > 20
]

print(valid_base_labels)

In [ ]:
def get_base_groups(df, base_label):
    single = df[df["subtle_labels"].apply(lambda x: x == [base_label])]

    multi = df[df["subtle_labels"].apply(
        lambda x: base_label in x and len(x) > 1
    )]

    return single, multi

In [ ]:
def get_centroid(df_sub):
    X = []
    for _, row in df_sub.iterrows():
        emb = load_embedding(row["rep_path"], TARGET_LAYER)
        if emb is not None:
            X.append(emb)
    return np.mean(X, axis=0)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def compute_directional_shift(single_df, multi_df, centroids):

    base_centroid = get_centroid(single_df)

    shifts = []

    for _, row in multi_df.iterrows():
        emb = load_embedding(row["rep_path"], TARGET_LAYER)
        if emb is None:
            continue

        labels = row["subtle_labels"]
        extra_labels = [l for l in labels if l != base_label]

        for l2 in extra_labels:
            if l2 not in centroids:
                continue

            direction = emb - base_centroid
            target_dir = centroids[l2] - base_centroid

            sim = cosine_similarity(
                direction.reshape(1, -1),
                target_dir.reshape(1, -1)
            )[0][0]

            shifts.append({
                "base": base_label,
                "added": l2,
                "similarity": sim
            })

    return shifts

In [ ]:
centroids = {}

for label in valid_base_labels:
    df_l = df[df["subtle_labels"].apply(lambda x: label in x)]
    centroids[label] = get_centroid(df_l)

In [ ]:
all_results = []

for base_label in valid_base_labels:
    single_df, multi_df = get_base_groups(df, base_label)

    if len(single_df) < 20 or len(multi_df) < 20:
        continue

    shifts = compute_directional_shift(single_df, multi_df, centroids)
    all_results.extend(shifts)

import pandas as pd
results_df = pd.DataFrame(all_results)

print(results_df.groupby(["base", "added"])["similarity"].mean())

In [ ]:
results_df.groupby("added")["similarity"].std()

In [ ]:
df

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

BASES = ["anger", "confusion", "neutral", "sadness", "surprise"]
embedding_cache = {}

for path in df["rep_path"].unique():

    emb = load_embedding(path, TARGET_LAYER)

    if emb is not None:
        embedding_cache[path] = emb
# =========================================================
# YOUR EXISTING CENTROID LOADER
# =========================================================

def get_centroid(df_sub):
    X = []

    for _, row in df_sub.iterrows():

        emb = embedding_cache.get(row["rep_path"])

        if emb is not None:
            X.append(emb)

    if len(X) == 0:
        return None

    return np.mean(X, axis=0)

# =========================================================
# BUILD CENTROIDS
# =========================================================

def build_centroids(df, base_emotions):

    centroids = {}

    for base in base_emotions:

        # -------------------------------------------------
        # BASE ONLY
        # -------------------------------------------------
        df_base = df[
            df["subtle_labels"].apply(
                lambda x: len(x) == 1 and base in x
            )
        ]

        base_centroid = get_centroid(df_base)

        if base_centroid is None:
            continue

        centroids[(base, None)] = base_centroid

        # -------------------------------------------------
        # BASE + ADDED
        # -------------------------------------------------
        for added in base_emotions:

            if added == base:
                continue

            df_pair = df[
                df["subtle_labels"].apply(
                    lambda x:
                        base in x and added in x
                )
            ]

            pair_centroid = get_centroid(df_pair)

            if pair_centroid is None:
                continue

            centroids[(base, added)] = pair_centroid

    return centroids

# =========================================================
# COMPUTE SHIFT MATRIX
# =========================================================

def compute_shift_matrix(centroids, base_emotions):

    rows = []

    for base in base_emotions:

        if (base, None) not in centroids:
            continue

        base_vec = centroids[(base, None)]

        for added in base_emotions:

            if added == base:
                continue

            if (base, added) not in centroids:
                continue

            pair_vec = centroids[(base, added)]

            sim = cosine_similarity(
                base_vec.reshape(1, -1),
                pair_vec.reshape(1, -1)
            )[0][0]

            rows.append({
                "base": base,
                "added": added,
                "similarity": sim
            })

    return pd.DataFrame(rows)

# =========================================================
# REAL RESULT
# =========================================================

real_centroids = build_centroids(df, BASES)

real_df = compute_shift_matrix(real_centroids, BASES)

print("\n=== REAL RESULT ===")
print(real_df)

real_mean = real_df["similarity"].mean()

print("\nREAL MEAN:")
print(real_mean)

# =========================================================
# RANDOM BASELINE
# =========================================================

N_PERM = 100
random_means = []

for i in range(N_PERM):

    print(f"Permutation {i+1}/{N_PERM}")

    df_rand = df.copy()

    shuffled = (
        df_rand["subtle_labels"]
        .sample(frac=1, replace=False)
        .values
    )

    df_rand["subtle_labels"] = shuffled

    rand_centroids = build_centroids(df_rand, BASES)

    rand_df = compute_shift_matrix(rand_centroids, BASES)

    if len(rand_df) == 0:
        continue

    random_means.append(
        rand_df["similarity"].mean()
    )

random_means = np.array(random_means)

# =========================================================
# FINAL STATS
# =========================================================

print("\n=== RANDOM BASELINE ===")
print("Mean:", random_means.mean())
print("Std :", random_means.std())

p_value = np.mean(random_means >= real_mean)

print("\n=== SIGNIFICANCE ===")
print("Real Mean :", real_mean)
print("P-value   :", p_value)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

BASES = ["anger", "confusion", "neutral", "sadness", "surprise"]

# =========================================================
# GET CENTROID
# =========================================================

def get_centroid(df_sub):

    X = []

    for _, row in df_sub.iterrows():

        emb = embedding_cache.get(row["rep_path"])

        if emb is not None:
            X.append(emb)

    if len(X) == 0:
        return None

    return np.mean(X, axis=0)

# =========================================================
# BUILD SHIFT VECTORS
# =========================================================

def build_shift_vectors(df, base_emotions):

    shifts = {}

    for base in base_emotions:

        # -------------------------------------------------
        # BASE ONLY
        # -------------------------------------------------
        df_base = df[
            df["subtle_labels"].apply(
                lambda x: len(x) == 1 and base in x
            )
        ]

        base_centroid = get_centroid(df_base)

        if base_centroid is None:
            continue

        # -------------------------------------------------
        # BASE + ADDED
        # -------------------------------------------------
        for added in base_emotions:

            if added == base:
                continue

            df_pair = df[
                df["subtle_labels"].apply(
                    lambda x:
                        base in x and added in x
                )
            ]

            pair_centroid = get_centroid(df_pair)

            if pair_centroid is None:
                continue

            # =============================================
            # SHIFT VECTOR
            # =============================================
            shift = pair_centroid - base_centroid

            shifts[(base, added)] = shift

    return shifts

# =========================================================
# COMPUTE SHIFT MAGNITUDES
# =========================================================

def compute_shift_magnitudes(shifts):

    rows = []

    for (base, added), vec in shifts.items():

        magnitude = np.linalg.norm(vec)

        rows.append({
            "base": base,
            "added": added,
            "magnitude": magnitude
        })

    return pd.DataFrame(rows)

# =========================================================
# REAL SHIFTS
# =========================================================

real_shifts = build_shift_vectors(df, BASES)

real_df = compute_shift_magnitudes(real_shifts)

print("\n=== REAL SHIFT MAGNITUDES ===")
print(real_df)

real_mean = real_df["magnitude"].mean()

print("\nREAL MEAN SHIFT:")
print(real_mean)

# =========================================================
# RANDOM BASELINE
# =========================================================

N_PERM = 100
random_means = []

for i in range(N_PERM):

    print(f"Permutation {i+1}/{N_PERM}")

    df_rand = df.copy()

    shuffled = (
        df_rand["subtle_labels"]
        .sample(frac=1, replace=False)
        .values
    )

    df_rand["subtle_labels"] = shuffled

    rand_shifts = build_shift_vectors(df_rand, BASES)

    rand_df = compute_shift_magnitudes(rand_shifts)

    if len(rand_df) == 0:
        continue

    random_means.append(
        rand_df["magnitude"].mean()
    )

random_means = np.array(random_means)

# =========================================================
# FINAL STATS
# =========================================================

print("\n=== RANDOM BASELINE ===")
print("Mean:", random_means.mean())
print("Std :", random_means.std())

p_value = np.mean(random_means >= real_mean)

print("\n=== SIGNIFICANCE ===")
print("Real Mean :", real_mean)
print("P-value   :", p_value)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.stats import gaussian_kde

# =========================================================
# SETTINGS
# =========================================================

BASE = "anger"
ADDED = "confusion"

# =========================================================
# FILTER
# =========================================================

df_base = df[
    df["subtle_labels"].apply(
        lambda x: len(x) == 1 and BASE in x
    )
]

df_mixed = df[
    df["subtle_labels"].apply(
        lambda x:
            BASE in x and ADDED in x
    )
]

# =========================================================
# LOAD EMBEDDINGS
# =========================================================

def collect_embeddings(df_sub):

    X = []

    for _, row in df_sub.iterrows():

        emb = embedding_cache.get(row["rep_path"])

        if emb is not None:
            X.append(emb)

    return np.stack(X)

X_base = collect_embeddings(df_base)
X_mixed = collect_embeddings(df_mixed)

# =========================================================
# STANDARDIZE
# =========================================================

X_all = np.concatenate([X_base, X_mixed], axis=0)

scaler = StandardScaler()

X_all = scaler.fit_transform(X_all)

X_base = X_all[:len(X_base)]
X_mixed = X_all[len(X_base):]

# =========================================================
# PCA
# =========================================================

pca = PCA(n_components=2)

X_all_2d = pca.fit_transform(
    np.concatenate([X_base, X_mixed], axis=0)
)

base_2d = X_all_2d[:len(X_base)]
mixed_2d = X_all_2d[len(X_base):]

# =========================================================
# CENTROIDS
# =========================================================

centroid_base = base_2d.mean(axis=0)
centroid_mixed = mixed_2d.mean(axis=0)

# =========================================================
# KDE FUNCTION
# =========================================================

def draw_kde(points, color, label):

    x = points[:, 0]
    y = points[:, 1]

    kde = gaussian_kde([x, y])

    xmin, xmax = x.min()-1, x.max()+1
    ymin, ymax = y.min()-1, y.max()+1

    xx, yy = np.meshgrid(
        np.linspace(xmin, xmax, 200),
        np.linspace(ymin, ymax, 200)
    )

    zz = kde(np.vstack([xx.ravel(), yy.ravel()]))
    zz = zz.reshape(xx.shape)

    plt.contourf(
        xx,
        yy,
        zz,
        levels=6,
        alpha=0.28,
        colors=[color]
    )

    plt.contour(
        xx,
        yy,
        zz,
        levels=6,
        colors=color,
        linewidths=1
    )

# =========================================================
# PLOT
# =========================================================

plt.figure(figsize=(9, 7))

# ---------------------------------------------------------
# density blobs
# ---------------------------------------------------------

draw_kde(
    base_2d,
    "red",
    f"{BASE}"
)

draw_kde(
    mixed_2d,
    "purple",
    f"{BASE}+{ADDED}"
)

# ---------------------------------------------------------
# centroids
# ---------------------------------------------------------

plt.scatter(
    centroid_base[0],
    centroid_base[1],
    s=450,
    color="red",
    edgecolors="black",
    linewidths=2,
    zorder=10
)

plt.scatter(
    centroid_mixed[0],
    centroid_mixed[1],
    s=450,
    color="purple",
    edgecolors="black",
    linewidths=2,
    zorder=10
)

# ---------------------------------------------------------
# arrow
# ---------------------------------------------------------

plt.arrow(
    centroid_base[0],
    centroid_base[1],
    centroid_mixed[0] - centroid_base[0],
    centroid_mixed[1] - centroid_base[1],
    width=0.01,
    head_width=0.12,
    color="black",
    alpha=0.9,
    length_includes_head=True,
    zorder=9
)

# ---------------------------------------------------------
# labels
# ---------------------------------------------------------

plt.text(
    centroid_base[0],
    centroid_base[1],
    BASE,
    color="white",
    ha="center",
    va="center",
    fontweight="bold"
)

plt.text(
    centroid_mixed[0],
    centroid_mixed[1],
    f"{BASE}+{ADDED}",
    color="white",
    ha="center",
    va="center",
    fontweight="bold"
)

# =========================================================
# STYLE
# =========================================================

plt.title(
    f"Local Manifold Perturbation\n"
    f"{BASE} → {BASE}+{ADDED}",
    fontsize=16
)

plt.xlabel("PCA-1")
plt.ylabel("PCA-2")

plt.grid(alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Create a pivot table for the heatmap
pivot = real_df.pivot(index='base', columns='added', values='magnitude')

plt.figure(figsize=(10, 8))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlOrRd',
            cbar_kws={'label': 'Shift Magnitude'}, linewidths=0.5)
plt.title('Embedding Shift Magnitudes: Base → Base+Added Emotion', fontsize=14)
plt.xlabel('Added Emotion', fontsize=12)
plt.ylabel('Base Emotion', fontsize=12)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
plt.hist(random_means, bins=20, alpha=0.7, label='Random Permutations',
         color='gray', edgecolor='black')
plt.axvline(real_mean, color='red', linewidth=3, linestyle='--',
            label=f'Real Mean: {real_mean:.2f}')
plt.axvline(np.mean(random_means), color='blue', linewidth=2, linestyle='-',
            label=f'Random Mean: {np.mean(random_means):.2f}')

# Add significance annotation
plt.text(real_mean + 0.05, plt.ylim()[1]*0.9, f'p < 0.01',
         fontsize=14, fontweight='bold', color='red')

plt.xlabel('Mean Shift Magnitude', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.title('Real vs Random Shift Magnitudes\n(Permutation Test, n=100)', fontsize=14)
plt.legend()
plt.tight_layout()
plt.show()

# Create symmetric difference matrix
emotions = BASES
diff_matrix = np.zeros((len(emotions), len(emotions)))

for i, e1 in enumerate(emotions):
    for j, e2 in enumerate(emotions):
        if e1 != e2:
            # Get magnitude from real data
            mag1 = real_df[(real_df['base']==e1) & (real_df['added']==e2)]['magnitude'].values
            mag2 = real_df[(real_df['base']==e2) & (real_df['added']==e1)]['magnitude'].values

            if len(mag1) > 0 and len(mag2) > 0:
                diff_matrix[i, j] = mag1[0] - mag2[0]  # Asymmetry

plt.figure(figsize=(10, 8))
sns.heatmap(diff_matrix, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, xticklabels=emotions, yticklabels=emotions,
            cbar_kws={'label': 'Magnitude Difference (Base - Base)'})
plt.title('Asymmetry in Emotional Shifts\n(Positive = Base→Added > Added→Base)', fontsize=14)
plt.xlabel('Added Emotion', fontsize=12)
plt.ylabel('Base Emotion', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
import networkx as nx

# Create directed graph
G = nx.DiGraph()

# Add edges with weights
for _, row in real_df.iterrows():
    G.add_edge(row['base'], row['added'], weight=row['magnitude'])

# Normalize edge widths
max_width = real_df['magnitude'].max()
min_width = real_df['magnitude'].min()

plt.figure(figsize=(12, 10))
pos = nx.circular_layout(G)  # or try spring_layout, kamada_kawai_layout

# Draw nodes
nx.draw_networkx_nodes(G, pos, node_size=3000, node_color='lightblue',
                       edgecolors='black', linewidths=2)

# Draw edges with varying thickness
for (u, v, d) in G.edges(data=True):
    width = 1 + 5 * (d['weight'] - min_width) / (max_width - min_width)
    nx.draw_networkx_edges(G, pos, edgelist=[(u, v)], width=width,
                           alpha=0.6, edge_color='darkred',
                           connectionstyle='arc3,rad=0.1',
                           arrows=True, arrowsize=20)

nx.draw_networkx_labels(G, pos, font_size=10, font_weight='bold')

# Add edge labels
edge_labels = {(u, v): f'{d["weight"]:.1f}' for u, v, d in G.edges(data=True)}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=8)

plt.title('Emotion Shift Network\n(Directed edges show shift magnitude)', fontsize=14)
plt.axis('off')
plt.tight_layout()
plt.show()



In [ ]:
import random
from sklearn.metrics.pairwise import cosine_similarity

def compute_random_directional_shift(single_df, multi_df, centroids):

    base_centroid = get_centroid(single_df)

    all_labels = list(centroids.keys())

    shifts = []

    for _, row in multi_df.iterrows():

        emb = load_embedding(row["rep_path"], TARGET_LAYER)
        if emb is None:
            continue

        labels = row["subtle_labels"]

        # actual added labels
        extra_labels = [l for l in labels if l != base_label]

        for _ in extra_labels:

            # random label (excluding base)
            rand_label = random.choice(
                [l for l in all_labels if l != base_label]
            )

            direction = emb - base_centroid

            target_dir = (
                centroids[rand_label] - base_centroid
            )

            sim = cosine_similarity(
                direction.reshape(1, -1),
                target_dir.reshape(1, -1)
            )[0][0]

            shifts.append({
                "base": base_label,
                "random_added": rand_label,
                "similarity": sim
            })

    return shifts

In [ ]:
random_results = []

for base_label in valid_base_labels:

    single_df, multi_df = get_base_groups(df, base_label)

    if len(single_df) < 20 or len(multi_df) < 20:
        continue

    shifts = compute_random_directional_shift(
        single_df,
        multi_df,
        centroids
    )

    random_results.extend(shifts)

random_df = pd.DataFrame(random_results)

In [ ]:
random_df

In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=random_df)

In [ ]:
import pandas as pd
import re

# =========================
# LOAD
# =========================
df = pd.read_csv("/content/drive/MyDrive/subprobe/task3_prompt_subset_v3_flamingo/results_partial.csv")

# =========================
# LABEL SETUP
# =========================

# Full valid label set (from your table)
VALID_LABELS = {
    "anger", "annoyance", "disapproval",
    "disgust",
    "fear", "nervousness",
    "joy", "amusement", "excitement", "pride", "optimism", "love",
    "neutral", "confusion", "curiosity", "realization", "surprise",
    "sadness", "disappointment", "remorse", "grief"
}

# Basic-7
BASIC_7 = {
    "anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"
}

# =========================
# STRICT EXTRACTION
# =========================

def extract_exact_label(text):
    if pd.isna(text):
        return "OOD"

    text = str(text).lower()

    # tokenize
    tokens = re.findall(r"[a-z]+", text)

    for t in tokens:
        if t in VALID_LABELS:
            return t

    return "OOD"

df["pred_strict"] = df["raw"].apply(extract_exact_label)

# =========================
# TARGET TYPE (basic vs fine)
# =========================

def label_type(label):
    if label in BASIC_7:
        return "basic"
    else:
        return "fine"

df["target_type"] = df["target"].apply(label_type)

# =========================
# METRICS
# =========================

# Accuracy
df["correct"] = df["pred_strict"] == df["target"]

# Valid prediction (not OOD)
df["valid"] = df["pred_strict"] != "OOD"

# =========================
# OVERALL RESULTS
# =========================

print("\n=== Accuracy by Prompt ===")
print(df.groupby("prompt_type")["correct"].mean())

print("\n=== OOD Rate by Prompt ===")
print((~df["valid"]).groupby(df["prompt_type"]).mean())

print("\n=== Valid Rate by Prompt ===")
print(df.groupby("prompt_type")["valid"].mean())

# =========================
# SPLIT: BASIC vs FINE
# =========================

print("\n=== Accuracy (Prompt × Target Type) ===")
print(df.groupby(["prompt_type", "target_type"])["correct"].mean())

print("\n=== OOD Rate (Prompt × Target Type) ===")
print((~df["valid"]).groupby([df["prompt_type"], df["target_type"]]).mean())

# =========================
# OPTIONAL: LABEL DIVERSITY
# =========================

print("\n=== Label Diversity ===")
print(df.groupby("prompt_type")["pred_strict"].nunique())

# =========================
# OPTIONAL: SAVE RESULTS
# =========================

df.to_csv("elicitation_analysis_results.csv", index=False)

In [ ]:
import pandas as pd
import re

# =========================
# LOAD
# =========================
df = pd.read_csv("/content/drive/MyDrive/subprobe/task3_prompt_subset_v3_qwen/results_partial.csv")

# =========================
# LABEL SETUP
# =========================

# Full valid label set (from your table)
VALID_LABELS = {
    "anger", "annoyance", "disapproval",
    "disgust",
    "fear", "nervousness",
    "joy", "amusement", "excitement", "pride", "optimism", "love",
    "neutral", "confusion", "curiosity", "realization", "surprise",
    "sadness", "disappointment", "remorse", "grief"
}

# Basic-7
BASIC_7 = {
    "anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"
}

# =========================
# STRICT EXTRACTION
# =========================

def extract_exact_label(text):
    if pd.isna(text):
        return "OOD"

    text = str(text).lower()

    # tokenize
    tokens = re.findall(r"[a-z]+", text)

    for t in tokens:
        if t in VALID_LABELS:
            return t

    return "OOD"

df["pred_strict"] = df["raw"].apply(extract_exact_label)

# =========================
# TARGET TYPE (basic vs fine)
# =========================

def label_type(label):
    if label in BASIC_7:
        return "basic"
    else:
        return "fine"

df["target_type"] = df["target"].apply(label_type)

# =========================
# METRICS
# =========================

# Accuracy
df["correct"] = df["pred_strict"] == df["target"]

# Valid prediction (not OOD)
df["valid"] = df["pred_strict"] != "OOD"

# =========================
# OVERALL RESULTS
# =========================

print("\n=== Accuracy by Prompt ===")
print(df.groupby("prompt_type")["correct"].mean())

print("\n=== OOD Rate by Prompt ===")
print((~df["valid"]).groupby(df["prompt_type"]).mean())

print("\n=== Valid Rate by Prompt ===")
print(df.groupby("prompt_type")["valid"].mean())

# =========================
# SPLIT: BASIC vs FINE
# =========================

print("\n=== Accuracy (Prompt × Target Type) ===")
print(df.groupby(["prompt_type", "target_type"])["correct"].mean())

print("\n=== OOD Rate (Prompt × Target Type) ===")
print((~df["valid"]).groupby([df["prompt_type"], df["target_type"]]).mean())

# =========================
# OPTIONAL: LABEL DIVERSITY
# =========================

print("\n=== Label Diversity ===")
print(df.groupby("prompt_type")["pred_strict"].nunique())

# =========================
# OPTIONAL: SAVE RESULTS
# =========================

df.to_csv("elicitation_analysis_results.csv", index=False)

In [ ]:
BASIC_7 = {"anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"}
import pandas as pd
import ast

df_labels = pd.read_csv("/content/drive/MyDrive/subprobe/task3_full_run_flamingo/task3_results.csv")
df_pred = pd.read_csv("/content/drive/MyDrive/subprobe/task3_prompt_subset_v3_flamingo/results_partial.csv")

# =========================
# PARSE LABELS
# =========================
def parse_labels(x):
    try:
        return ast.literal_eval(x)
    except:
        return []

df_labels["label_list"] = df_labels["labels"].apply(parse_labels)

# =========================
# DEFINE BASIC LABELS
# =========================
BASIC_7 = {"anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"}

# =========================
# DEFINE SUBTLE TYPES
# =========================
def categorize(labels):
    if len(labels) <= 1:
        return "single"

    has_basic = any(l in BASIC_7 for l in labels)
    has_fine = any(l not in BASIC_7 for l in labels)

    if has_fine and not has_basic:
        return "pure_subtle"
    elif has_fine and has_basic:
        return "mixed"
    else:
        return "basic_only"

df_labels["subtle_type"] = df_labels["label_list"].apply(categorize)

# =========================
# STRICT LABEL SET
# =========================
VALID_LABELS = {
    "anger", "annoyance", "disapproval",
    "disgust",
    "fear", "nervousness",
    "joy", "amusement", "excitement", "pride", "optimism", "love",
    "neutral", "confusion", "curiosity", "realization", "surprise",
    "sadness", "disappointment", "remorse", "grief"
}

# =========================
# EXTRACT FROM RAW
# =========================
def extract_exact_label(text):
    if pd.isna(text):
        return "OOD"

    text = str(text).lower()
    tokens = re.findall(r"[a-z]+", text)

    for t in tokens:
        if t in VALID_LABELS:
            return t

    return "OOD"

df_pred["pred_strict"] = df_pred["raw"].apply(extract_exact_label)

# =========================
# MERGE
# =========================
df = pd.merge(
    df_pred,
    df_labels[["id", "subtle_type"]],
    on="id",
    how="inner"
)

# =========================
# METRICS
# =========================
df["correct"] = df["pred_strict"] == df["target"]
df["valid"] = df["pred_strict"] != "OOD"

# =========================
# OPTIONAL: FILTER
# =========================
df = df[df["subtle_type"] != "single"]

# remove neutral targets
df = df[df["target"] != "neutral"]

# =========================
# SANITY CHECK
# =========================
print("\nSubtle type distribution:")
print(df["subtle_type"].value_counts())

# =========================
# RESULTS
# =========================

print("\n=== Accuracy (Prompt × Subtle Type) ===")
print(df.groupby(["prompt_type", "subtle_type"])["correct"].mean())

print("\n=== OOD Rate (Prompt × Subtle Type) ===")
print((~df["valid"]).groupby([df["prompt_type"], df["subtle_type"]]).mean())

print("\n=== Valid Rate (Prompt × Subtle Type) ===")
print(df.groupby(["prompt_type", "subtle_type"])["valid"].mean())

In [ ]:
BASIC_7 = {"anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"}
import pandas as pd
import ast

df_labels = pd.read_csv("/content/drive/MyDrive/subprobe/task3_full_run_qwen/task3_results.csv")
df_pred = pd.read_csv("/content/drive/MyDrive/subprobe/task3_prompt_subset_v3_qwen/results_partial.csv")

# =========================
# PARSE LABELS
# =========================
def parse_labels(x):
    try:
        return ast.literal_eval(x)
    except:
        return []

df_labels["label_list"] = df_labels["labels"].apply(parse_labels)

# =========================
# DEFINE BASIC LABELS
# =========================
BASIC_7 = {"anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"}

# =========================
# DEFINE SUBTLE TYPES
# =========================
def categorize(labels):
    if len(labels) <= 1:
        return "single"

    has_basic = any(l in BASIC_7 for l in labels)
    has_fine = any(l not in BASIC_7 for l in labels)

    if has_fine and not has_basic:
        return "pure_subtle"
    elif has_fine and has_basic:
        return "mixed"
    else:
        return "basic_only"

df_labels["subtle_type"] = df_labels["label_list"].apply(categorize)

# =========================
# STRICT LABEL SET
# =========================
VALID_LABELS = {
    "anger", "annoyance", "disapproval",
    "disgust",
    "fear", "nervousness",
    "joy", "amusement", "excitement", "pride", "optimism", "love",
    "neutral", "confusion", "curiosity", "realization", "surprise",
    "sadness", "disappointment", "remorse", "grief"
}

# =========================
# EXTRACT FROM RAW
# =========================
def extract_exact_label(text):
    if pd.isna(text):
        return "OOD"

    text = str(text).lower()
    tokens = re.findall(r"[a-z]+", text)

    for t in tokens:
        if t in VALID_LABELS:
            return t

    return "OOD"

df_pred["pred_strict"] = df_pred["raw"].apply(extract_exact_label)

# =========================
# MERGE
# =========================
df = pd.merge(
    df_pred,
    df_labels[["id", "subtle_type"]],
    on="id",
    how="inner"
)

# =========================
# METRICS
# =========================
df["correct"] = df["pred_strict"] == df["target"]
df["valid"] = df["pred_strict"] != "OOD"

# =========================
# OPTIONAL: FILTER
# =========================
df = df[df["subtle_type"] != "single"]

# remove neutral targets
df = df[df["target"] != "neutral"]

# =========================
# SANITY CHECK
# =========================
print("\nSubtle type distribution:")
print(df["subtle_type"].value_counts())

# =========================
# RESULTS
# =========================

print("\n=== Accuracy (Prompt × Subtle Type) ===")
print(df.groupby(["prompt_type", "subtle_type"])["correct"].mean())

print("\n=== OOD Rate (Prompt × Subtle Type) ===")
print((~df["valid"]).groupby([df["prompt_type"], df["subtle_type"]]).mean())

print("\n=== Valid Rate (Prompt × Subtle Type) ===")
print(df.groupby(["prompt_type", "subtle_type"])["valid"].mean())

In [ ]:
import pandas as pd
import ast
import re

# =========================================================
# LOAD
# =========================================================

df_labels = pd.read_csv(
    "/content/drive/MyDrive/subprobe/task3_full_run_flamingo/task3_results.csv"
)

df_pred = pd.read_csv(
    "/content/drive/MyDrive/subprobe/task3_text_baseline_af3/final_results.csv"
)

# =========================================================
# PARSE LABELS
# =========================================================

def parse_labels(x):

    try:
        return ast.literal_eval(x)

    except:
        return []

df_labels["label_list"] = df_labels["labels"].apply(parse_labels)

# =========================================================
# BASIC LABELS
# =========================================================

BASIC_7 = {
    "anger",
    "disgust",
    "fear",
    "joy",
    "neutral",
    "sadness",
    "surprise"
}

# =========================================================
# SUBTLE TYPES
# =========================================================

def categorize(labels):

    if len(labels) <= 1:
        return "single"

    has_basic = any(l in BASIC_7 for l in labels)

    has_fine = any(l not in BASIC_7 for l in labels)

    if has_fine and not has_basic:
        return "pure_subtle"

    elif has_fine and has_basic:
        return "mixed"

    else:
        return "basic_only"

df_labels["subtle_type"] = df_labels["label_list"].apply(categorize)

# =========================================================
# VALID LABELS
# =========================================================

VALID_LABELS = {
    "anger", "annoyance", "disapproval",
    "disgust",
    "fear", "nervousness",
    "joy", "amusement", "excitement",
    "pride", "optimism", "love",
    "neutral",
    "confusion", "curiosity",
    "realization", "surprise",
    "sadness", "disappointment",
    "remorse", "grief"
}

# =========================================================
# STRICT LABEL EXTRACTION
# =========================================================

def extract_exact_label(text):

    if pd.isna(text):
        return "OOD"

    text = str(text).lower()

    tokens = re.findall(r"[a-z]+", text)

    for t in tokens:

        if t in VALID_LABELS:
            return t

    return "OOD"

# =========================================================
# PREDICTION CLEANUP
# =========================================================

# text baseline has:
# pred
# raw
# target
# no prompt_type column



df_pred["pred_strict"] = df_pred["raw"].apply(
    extract_exact_label
)

# =========================================================
# ADD FIXED PROMPT TYPE
# =========================================================

df_pred["prompt_type"] = "same_cluster_text"

# =========================================================
# MERGE
# =========================================================

df = pd.merge(
    df_pred,
    df_labels[["id", "subtle_type"]],
    on="id",
    how="inner"
)

# =========================================================
# METRICS
# =========================================================

df["correct"] = df["pred_strict"] == df["target"]

df["valid"] = df["pred_strict"] != "OOD"

# =========================================================
# OPTIONAL FILTER
# =========================================================

df = df[df["subtle_type"] != "single"]

# remove neutral targets
df = df[df["target"] != "neutral"]

# =========================================================
# SANITY CHECK
# =========================================================

print("\nSubtle type distribution:")
print(df["subtle_type"].value_counts())

# =========================================================
# RESULTS
# =========================================================

print("\n=== Accuracy ===")
print(
    df.groupby(["prompt_type", "subtle_type"])["correct"].mean()
)

print("\n=== OOD Rate ===")
print(
    (~df["valid"]).groupby(
        [df["prompt_type"], df["subtle_type"]]
    ).mean()
)

print("\n=== Valid Rate ===")
print(
    df.groupby(["prompt_type", "subtle_type"])["valid"].mean()
)

# =========================================================
# COUNTS
# =========================================================

print("\n=== Counts ===")

print(
    df.groupby(["prompt_type", "subtle_type"])
      .size()
)

# =========================================================
# RANDOM BASELINE
# =========================================================

print("\n=== Random Guess Baseline ===")
print("3-way MCQ chance level = 0.3333")

In [ ]:
import pandas as pd
import ast
import re

# =========================================================
# LOAD
# =========================================================

df_labels = pd.read_csv(
    "/content/drive/MyDrive/subprobe/task3_full_run_qwen/task3_results.csv"
)

df_pred = pd.read_csv(
    "/content/drive/MyDrive/subprobe/task3_text_baseline_qwen/text_only_same_cluster_results.csv"
)

# =========================================================
# PARSE LABELS
# =========================================================

def parse_labels(x):

    try:
        return ast.literal_eval(x)

    except:
        return []

df_labels["label_list"] = df_labels["labels"].apply(parse_labels)

# =========================================================
# BASIC LABELS
# =========================================================

BASIC_7 = {
    "anger",
    "disgust",
    "fear",
    "joy",
    "neutral",
    "sadness",
    "surprise"
}

# =========================================================
# SUBTLE TYPES
# =========================================================

def categorize(labels):

    if len(labels) <= 1:
        return "single"

    has_basic = any(l in BASIC_7 for l in labels)

    has_fine = any(l not in BASIC_7 for l in labels)

    if has_fine and not has_basic:
        return "pure_subtle"

    elif has_fine and has_basic:
        return "mixed"

    else:
        return "basic_only"

df_labels["subtle_type"] = df_labels["label_list"].apply(categorize)

# =========================================================
# VALID LABELS
# =========================================================

VALID_LABELS = {
    "anger", "annoyance", "disapproval",
    "disgust",
    "fear", "nervousness",
    "joy", "amusement", "excitement",
    "pride", "optimism", "love",
    "neutral",
    "confusion", "curiosity",
    "realization", "surprise",
    "sadness", "disappointment",
    "remorse", "grief"
}

# =========================================================
# STRICT LABEL EXTRACTION
# =========================================================

def extract_exact_label(text):

    if pd.isna(text):
        return "OOD"

    text = str(text).lower()

    tokens = re.findall(r"[a-z]+", text)

    for t in tokens:

        if t in VALID_LABELS:
            return t

    return "OOD"

# =========================================================
# PREDICTION CLEANUP
# =========================================================

# text baseline has:
# pred
# raw
# target
# no prompt_type column

df_pred["pred_strict"] = df_pred["raw"].apply(
    extract_exact_label
)

# =========================================================
# ADD FIXED PROMPT TYPE
# =========================================================

df_pred["prompt_type"] = "same_cluster_text"

# =========================================================
# MERGE
# =========================================================

df = pd.merge(
    df_pred,
    df_labels[["id", "subtle_type"]],
    on="id",
    how="inner"
)

# =========================================================
# METRICS
# =========================================================

df["correct"] = df["pred_strict"] == df["target"]

df["valid"] = df["pred_strict"] != "OOD"

# =========================================================
# OPTIONAL FILTER
# =========================================================

df = df[df["subtle_type"] != "single"]

# remove neutral targets
df = df[df["target"] != "neutral"]

# =========================================================
# SANITY CHECK
# =========================================================

print("\nSubtle type distribution:")
print(df["subtle_type"].value_counts())

# =========================================================
# RESULTS
# =========================================================

print("\n=== Accuracy ===")
print(
    df.groupby(["prompt_type", "subtle_type"])["correct"].mean()
)

print("\n=== OOD Rate ===")
print(
    (~df["valid"]).groupby(
        [df["prompt_type"], df["subtle_type"]]
    ).mean()
)

print("\n=== Valid Rate ===")
print(
    df.groupby(["prompt_type", "subtle_type"])["valid"].mean()
)

# =========================================================
# COUNTS
# =========================================================

print("\n=== Counts ===")

print(
    df.groupby(["prompt_type", "subtle_type"])
      .size()
)

# =========================================================
# RANDOM BASELINE
# =========================================================

print("\n=== Random Guess Baseline ===")
print("3-way MCQ chance level = 0.3333")

In [ ]:
import pandas as pd
import ast
import re

# =========================================================
# LOAD
# =========================================================

df_labels = pd.read_csv(
    "/content/drive/MyDrive/subprobe/task3_full_run_flamingo/task3_results.csv"
)

df_pred = pd.read_csv(
    "/content/drive/MyDrive/subprobe/task3_prompt_subset_v3_flamingo_multimodal/final_merged_results.csv"
)

# =========================================================
# PARSE LABELS
# =========================================================

def parse_labels(x):

    try:
        return ast.literal_eval(x)

    except:
        return []

df_labels["label_list"] = df_labels["labels"].apply(parse_labels)

# =========================================================
# BASIC LABELS
# =========================================================

BASIC_7 = {
    "anger",
    "disgust",
    "fear",
    "joy",
    "neutral",
    "sadness",
    "surprise"
}

# =========================================================
# SUBTLE TYPES
# =========================================================

def categorize(labels):

    if len(labels) <= 1:
        return "single"

    has_basic = any(l in BASIC_7 for l in labels)

    has_fine = any(l not in BASIC_7 for l in labels)

    if has_fine and not has_basic:
        return "pure_subtle"

    elif has_fine and has_basic:
        return "mixed"

    else:
        return "basic_only"

df_labels["subtle_type"] = df_labels["label_list"].apply(categorize)

# =========================================================
# VALID LABELS
# =========================================================

VALID_LABELS = {
    "anger", "annoyance", "disapproval",
    "disgust",
    "fear", "nervousness",
    "joy", "amusement", "excitement",
    "pride", "optimism", "love",
    "neutral",
    "confusion", "curiosity",
    "realization", "surprise",
    "sadness", "disappointment",
    "remorse", "grief"
}

# =========================================================
# STRICT LABEL EXTRACTION
# =========================================================

def extract_exact_label(text):

    if pd.isna(text):
        return "OOD"

    text = str(text).lower()

    tokens = re.findall(r"[a-z]+", text)

    for t in tokens:

        if t in VALID_LABELS:
            return t

    return "OOD"

# =========================================================
# PREDICTION CLEANUP
# =========================================================

# text baseline has:
# pred
# raw
# target
# no prompt_type column

df_pred["pred_strict"] = df_pred["raw"].apply(
    extract_exact_label
)

# =========================================================
# ADD FIXED PROMPT TYPE
# =========================================================

df_pred["prompt_type"] = "same_cluster_multi"

# =========================================================
# MERGE
# =========================================================

df = pd.merge(
    df_pred,
    df_labels[["id", "subtle_type"]],
    on="id",
    how="inner"
)

# =========================================================
# METRICS
# =========================================================

df["correct"] = df["pred_strict"] == df["target"]

df["valid"] = df["pred_strict"] != "OOD"

# =========================================================
# OPTIONAL FILTER
# =========================================================

df = df[df["subtle_type"] != "single"]

# remove neutral targets
df = df[df["target"] != "neutral"]

# =========================================================
# SANITY CHECK
# =========================================================

print("\nSubtle type distribution:")
print(df["subtle_type"].value_counts())

# =========================================================
# RESULTS
# =========================================================

print("\n=== Accuracy ===")
print(
    df.groupby(["prompt_type", "subtle_type"])["correct"].mean()
)

print("\n=== OOD Rate ===")
print(
    (~df["valid"]).groupby(
        [df["prompt_type"], df["subtle_type"]]
    ).mean()
)

print("\n=== Valid Rate ===")
print(
    df.groupby(["prompt_type", "subtle_type"])["valid"].mean()
)

# =========================================================
# COUNTS
# =========================================================

print("\n=== Counts ===")

print(
    df.groupby(["prompt_type", "subtle_type"])
      .size()
)

# =========================================================
# RANDOM BASELINE
# =========================================================

print("\n=== Random Guess Baseline ===")
print("3-way MCQ chance level = 0.3333")

In [ ]:
import pandas as pd
import ast
import re

# =========================================================
# LOAD
# =========================================================

df_labels = pd.read_csv(
    "/content/drive/MyDrive/subprobe/task3_full_run_qwen/task3_results.csv"
)

df_pred = pd.read_csv(
    "/content/drive/MyDrive/subprobe/task3_prompt_subset_v3_qwen_multimodal/final_merged_results.csv"
)

# =========================================================
# PARSE LABELS
# =========================================================

def parse_labels(x):

    try:
        return ast.literal_eval(x)

    except:
        return []

df_labels["label_list"] = df_labels["labels"].apply(parse_labels)

# =========================================================
# BASIC LABELS
# =========================================================

BASIC_7 = {
    "anger",
    "disgust",
    "fear",
    "joy",
    "neutral",
    "sadness",
    "surprise"
}

# =========================================================
# SUBTLE TYPES
# =========================================================

def categorize(labels):

    if len(labels) <= 1:
        return "single"

    has_basic = any(l in BASIC_7 for l in labels)

    has_fine = any(l not in BASIC_7 for l in labels)

    if has_fine and not has_basic:
        return "pure_subtle"

    elif has_fine and has_basic:
        return "mixed"

    else:
        return "basic_only"

df_labels["subtle_type"] = df_labels["label_list"].apply(categorize)

# =========================================================
# VALID LABELS
# =========================================================

VALID_LABELS = {
    "anger", "annoyance", "disapproval",
    "disgust",
    "fear", "nervousness",
    "joy", "amusement", "excitement",
    "pride", "optimism", "love",
    "neutral",
    "confusion", "curiosity",
    "realization", "surprise",
    "sadness", "disappointment",
    "remorse", "grief"
}

# =========================================================
# STRICT LABEL EXTRACTION
# =========================================================

def extract_exact_label(text):

    if pd.isna(text):
        return "OOD"

    text = str(text).lower()

    tokens = re.findall(r"[a-z]+", text)

    for t in tokens:

        if t in VALID_LABELS:
            return t

    return "OOD"

# =========================================================
# PREDICTION CLEANUP
# =========================================================

# text baseline has:
# pred
# raw
# target
# no prompt_type column

df_pred["pred_strict"] = df_pred["raw"].apply(
    extract_exact_label
)

# =========================================================
# ADD FIXED PROMPT TYPE
# =========================================================

df_pred["prompt_type"] = "same_cluster_Multi"

# =========================================================
# MERGE
# =========================================================

df = pd.merge(
    df_pred,
    df_labels[["id", "subtle_type"]],
    on="id",
    how="inner"
)

# =========================================================
# METRICS
# =========================================================

df["correct"] = df["pred_strict"] == df["target"]

df["valid"] = df["pred_strict"] != "OOD"

# =========================================================
# OPTIONAL FILTER
# =========================================================

df = df[df["subtle_type"] != "single"]

# remove neutral targets
df = df[df["target"] != "neutral"]

# =========================================================
# SANITY CHECK
# =========================================================

print("\nSubtle type distribution:")
print(df["subtle_type"].value_counts())

# =========================================================
# RESULTS
# =========================================================

print("\n=== Accuracy ===")
print(
    df.groupby(["prompt_type", "subtle_type"])["correct"].mean()
)

print("\n=== OOD Rate ===")
print(
    (~df["valid"]).groupby(
        [df["prompt_type"], df["subtle_type"]]
    ).mean()
)

print("\n=== Valid Rate ===")
print(
    df.groupby(["prompt_type", "subtle_type"])["valid"].mean()
)

# =========================================================
# COUNTS
# =========================================================

print("\n=== Counts ===")

print(
    df.groupby(["prompt_type", "subtle_type"])
      .size()
)

# =========================================================
# RANDOM BASELINE
# =========================================================

print("\n=== Random Guess Baseline ===")
print("3-way MCQ chance level = 0.3333")

In [ ]:
import pandas as pd
import ast
import re
from collections import Counter

# =========================================================
# LOAD
# =========================================================

df_labels = pd.read_csv(
    "/content/drive/MyDrive/subprobe/task3_full_run_flamingo/task3_results.csv"
)

df_pred = pd.read_csv(
    "/content/drive/MyDrive/subprobe/task3_prompt_subset_v3_flamingo/results_partial.csv"
)

# =========================================================
# PARSE LABELS
# =========================================================

def parse_labels(x):
    try:
        return ast.literal_eval(x)
    except:
        return []

def parse_counts(x):
    try:
        return ast.literal_eval(x)
    except:
        return {}

df_labels["label_list"] = df_labels["labels"].apply(parse_labels)
df_labels["label_counts_dict"] = df_labels["label_counts"].apply(parse_counts)

# =========================================================
# BASIC EMOTIONS
# =========================================================

BASIC_7 = {
    "anger",
    "disgust",
    "fear",
    "joy",
    "neutral",
    "sadness",
    "surprise"
}

# =========================================================
# SUBTLE TYPE
# =========================================================

def categorize(labels):

    if len(labels) <= 1:
        return "single"

    has_basic = any(l in BASIC_7 for l in labels)
    has_fine = any(l not in BASIC_7 for l in labels)

    if has_fine and not has_basic:
        return "pure_subtle"

    elif has_fine and has_basic:
        return "mixed"

    else:
        return "basic_only"

df_labels["subtle_type"] = (
    df_labels["label_list"]
    .apply(categorize)
)

# =========================================================
# CONSENSUS HIERARCHY
# =========================================================

"""
Hierarchy:

1. strong_dominant
   top emotion > 0.5

2. dominant_salient
   top-ranked emotion regardless of percentage

3. secondary_salient
   second-ranked emotion

4. diffuse_trace
   all remaining emotions
"""

def build_consensus_hierarchy(count_dict):

    total = sum(count_dict.values())

    if total == 0:
        return {}

    # proportions
    proportions = {
        k: v / total
        for k, v in count_dict.items()
    }

    # sort descending
    sorted_items = sorted(
        proportions.items(),
        key=lambda x: x[1],
        reverse=True
    )

    hierarchy = {}

    # =====================================================
    # STRONG DOMINANT
    # =====================================================

    top_label, top_score = sorted_items[0]

    if top_score > 0.5:
        hierarchy[top_label] = "strong_dominant"

    else:
        hierarchy[top_label] = "dominant_salient"

    # =====================================================
    # SECONDARY
    # =====================================================

    if len(sorted_items) > 1:

        second_score = sorted_items[1][1]

        for label, score in sorted_items[1:]:

            # tied second place
            if score == second_score:
                hierarchy[label] = "secondary_salient"

    # =====================================================
    # REMAINING
    # =====================================================

    for label, _ in sorted_items:

        if label not in hierarchy:
            hierarchy[label] = "diffuse_trace"

    return hierarchy

# build hierarchy dict for each sample
df_labels["consensus_hierarchy"] = (
    df_labels["label_counts_dict"]
    .apply(build_consensus_hierarchy)
)

# =========================================================
# STRICT LABELS
# =========================================================

VALID_LABELS = {
    "anger",
    "annoyance",
    "approval",
    "amusement",
    "admiration",
    "caring",
    "confusion",
    "curiosity",
    "desire",
    "disappointment",
    "disapproval",
    "disgust",
    "embarrassment",
    "excitement",
    "fear",
    "gratitude",
    "grief",
    "joy",
    "love",
    "neutral",
    "nervousness",
    "optimism",
    "pride",
    "realization",
    "relief",
    "remorse",
    "sadness",
    "surprise"
}

# =========================================================
# EXTRACT LABEL
# =========================================================

def extract_exact_label(text):

    if pd.isna(text):
        return "OOD"

    text = str(text).lower()

    tokens = re.findall(r"[a-z]+", text)

    for t in tokens:

        if t in VALID_LABELS:
            return t

    return "OOD"

df_pred["pred_strict"] = (
    df_pred["raw"]
    .apply(extract_exact_label)
)

# =========================================================
# MERGE
# =========================================================

df = pd.merge(
    df_pred,
    df_labels[
        [
            "id",
            "subtle_type",
            "consensus_hierarchy"
        ]
    ],
    on="id",
    how="inner"
)

# =========================================================
# ASSIGN CONSENSUS LEVEL
# =========================================================

def lookup_consensus(row):

    pred = row["target"]

    hierarchy = row["consensus_hierarchy"]

    return hierarchy.get(pred, "diffuse_trace")

df["consensus_level"] = (
    df.apply(lookup_consensus, axis=1)
)

# =========================================================
# METRICS
# =========================================================

df["correct"] = (
    df["pred_strict"] == df["target"]
)

df["valid"] = (
    df["pred_strict"] != "OOD"
)

# =========================================================
# OPTIONAL FILTERS
# =========================================================

df = df[df["subtle_type"] != "single"]

df = df[df["target"] != "neutral"]

# =========================================================
# SANITY CHECK
# =========================================================

print("\n=== Subtle Type Distribution ===")
print(df["subtle_type"].value_counts())

print("\n=== Consensus Level Distribution ===")
print(df["consensus_level"].value_counts())

# =========================================================
# OVERALL PROMPT PERFORMANCE
# =========================================================

print("\n=== Accuracy by Prompt ===")
print(
    df.groupby("prompt_type")["correct"]
    .mean()
)

print("\n=== OOD Rate by Prompt ===")
print(
    (~df["valid"])
    .groupby(df["prompt_type"])
    .mean()
)

print("\n=== Valid Rate by Prompt ===")
print(
    df.groupby("prompt_type")["valid"]
    .mean()
)

# =========================================================
# PROMPT × SUBTLE TYPE
# =========================================================

print("\n=== Accuracy (Prompt × Subtle Type) ===")

print(
    df.groupby(
        ["prompt_type", "subtle_type"]
    )["correct"]
    .mean()
)

print("\n=== OOD Rate (Prompt × Subtle Type) ===")

print(
    (~df["valid"])
    .groupby(
        [df["prompt_type"], df["subtle_type"]]
    )
    .mean()
)

print("\n=== Valid Rate (Prompt × Subtle Type) ===")

print(
    df.groupby(
        ["prompt_type", "subtle_type"]
    )["valid"]
    .mean()
)

# =========================================================
# PROMPT × CONSENSUS LEVEL
# =========================================================

print("\n=== Accuracy (Prompt × Consensus Level) ===")

print(
    df.groupby(
        ["prompt_type", "consensus_level"]
    )["correct"]
    .mean()
)

print("\n=== OOD Rate (Prompt × Consensus Level) ===")

print(
    (~df["valid"])
    .groupby(
        [df["prompt_type"], df["consensus_level"]]
    )
    .mean()
)

print("\n=== Valid Rate (Prompt × Consensus Level) ===")

print(
    df.groupby(
        ["prompt_type", "consensus_level"]
    )["valid"]
    .mean()
)

# =========================================================
# LABEL DIVERSITY
# =========================================================

print("\n=== Label Diversity ===")

print(
    df.groupby("prompt_type")["pred_strict"]
    .nunique()
)